# Agent for S05E05: Master Controller

This notebook implements a master controller agent with the components discussed in the S05E05 lesson: Tool Registry, Routing Intelligence, Memory Triad, and Graduated Autonomy.

## Components:
1. Tool Registry - manages available tools
2. Routing Intelligence - decides which tools/agents to use
3. Memory Triad - short-term, long-term, and episodic memory
4. Graduated Autonomy - different levels of human oversight

In [ ]:
# Install required packages
!pip install openai chromadb

In [ ]:
from openai import OpenAI
import chromadb
from datetime import datetime
import json

class MasterController:
    def __init__(self, api_key):
        self.client = OpenAI(api_key=api_key)
        
        # Tool Registry
        self.tool_registry = {}
        self.register_core_tools()
        
        # Memory Triad
        self.short_term_memory = []
        self.chroma_client = chromadb.PersistentClient(path="./memory_db")
        self.long_term_memory = self.chroma_client.get_or_create_collection("long_term")
        self.episodic_memory = self.chroma_client.get_or_create_collection("episodic")
        
        # Autonomy levels
        self.autonomy_levels = {
            'read_only': 0,
            'supervised': 1,
            'trusted': 2,
            'full_auto': 3
        }
        self.current_autonomy = 'supervised'
    
    def register_core_tools(self):
        """Register core tools in the registry"""
        self.tool_registry = {
            'web_search': {
                'description': 'Search the web for information',
                'cost': 0.01,
                'limits': {'daily': 100},
                'function': self.mock_web_search
            },
            'file_read': {
                'description': 'Read content from a file',
                'cost': 0.001,
                'limits': {'daily': 1000},
                'function': self.mock_file_read
            },
            'code_execute': {
                'description': 'Execute code in a safe environment',
                'cost': 0.1,
                'limits': {'daily': 10},
                'function': self.mock_code_execute
            }
        }
    
    def mock_web_search(self, query):
        return f"Mock search results for: {query}"
    
    def mock_file_read(self, path):
        return f"Mock content from file: {path}"
    
    def mock_code_execute(self, code):
        return f"Mock execution result for: {code[:50]}..."
    
    def route_intelligence(self, task):
        """Decide which tools/agents to use for a task"""
        prompt = f"""
        Available tools: {list(self.tool_registry.keys())}
        Task: {task}
        Current autonomy: {self.current_autonomy}
        
        Decide:
        1. Which tools to use
        2. If human approval is needed
        3. Task complexity (simple/medium/complex)
        
        Return JSON with 'tools', 'needs_approval', 'complexity'
        """
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}]
        )
        
        try:
            decision = json.loads(response.choices[0].message.content)
            return decision
        except:
            return {'tools': ['web_search'], 'needs_approval': True, 'complexity': 'medium'}
    
    def update_memory(self, memory_type, content, metadata=None):
        """Update memory triad"""
        if memory_type == 'short_term':
            self.short_term_memory.append({
                'content': content,
                'timestamp': datetime.now().isoformat(),
                'metadata': metadata or {}
            })
            # Keep only last 10 items
            self.short_term_memory = self.short_term_memory[-10:]
        
        elif memory_type == 'long_term':
            self.long_term_memory.add(
                documents=[content],
                metadatas=[metadata or {}],
                ids=[f"lt_{datetime.now().timestamp()}"]
            )
        
        elif memory_type == 'episodic':
            self.episodic_memory.add(
                documents=[content],
                metadatas=[metadata or {}],
                ids=[f"ep_{datetime.now().timestamp()}"]
            )
    
    def retrieve_memory(self, query, memory_type):
        """Retrieve from memory triad"""
        if memory_type == 'short_term':
            return self.short_term_memory
        
        elif memory_type == 'long_term':
            results = self.long_term_memory.query(
                query_texts=[query],
                n_results=3
            )
            return results['documents']
        
        elif memory_type == 'episodic':
            results = self.episodic_memory.query(
                query_texts=[query],
                n_results=3
            )
            return results['documents']
    
    def check_autonomy(self, task_complexity):
        """Check if human approval is needed based on autonomy level"""
        autonomy_value = self.autonomy_levels[self.current_autonomy]
        
        if task_complexity == 'complex' and autonomy_value < 2:
            return True  # Needs approval
        elif task_complexity == 'medium' and autonomy_value < 1:
            return True
        else:
            return False
    
    def execute_task(self, task):
        """Execute task using master controller logic"""
        # Step 1: Route intelligence
        routing = self.route_intelligence(task)
        
        # Step 2: Check autonomy
        needs_approval = routing.get('needs_approval', False) or self.check_autonomy(routing.get('complexity', 'medium'))
        
        if needs_approval:
            print(f"Task requires approval: {task}")
            approval = input("Approve? (yes/no): ")
            if approval.lower() != 'yes':
                return "Task cancelled"
        
        # Step 3: Execute with tools
        results = []
        for tool_name in routing.get('tools', []):
            if tool_name in self.tool_registry:
                tool_info = self.tool_registry[tool_name]
                result = tool_info['function'](task)
                results.append(f"{tool_name}: {result}")
        
        final_result = "; ".join(results) if results else "No tools executed"
        
        # Step 4: Update memory
        self.update_memory('short_term', f"Task: {task}, Result: {final_result}")
        self.update_memory('episodic', f"Executed task: {task}", {'success': True})
        
        return final_result

# Initialize master controller (requires OpenAI API key)
# controller = MasterController(api_key='your-openai-key')

In [ ]:
# Example usage (uncomment and add API key)
# result = controller.execute_task("Search for Python tutorials")
# print(result)
# print("Short-term memory:", controller.retrieve_memory("", "short_term"))

In [ ]:
# Example with memory
# controller.update_memory('long_term', "Python is a programming language", {'topic': 'programming'})
# memories = controller.retrieve_memory("programming language", "long_term")
# print("Retrieved memories:", memories)